# EO-Benchmark Hub — BigEarthNet-S2 RGB (10%, 19 classes) Baseline

Multi-label land-cover classification on **BigEarthNet v2.0 (reBEN)**, reduced to a Colab-friendly benchmark: a deterministic **10% subset**, **RGB bands only** (B04/B03/B02), the **official geographic splits** preserved, and the **19-class** nomenclature.

**This notebook has two halves:**
- **Section 3 — PREP (runs once, ~1–2 h):** downloads the official 59 GiB archive to Colab's local disk, stream-extracts *only* our selected patches as 120×120 RGB PNGs, and saves a ~1–2 GB subset zip + split manifest to your Drive. It auto-skips on every later run.
- **Sections 4+ — TRAIN (every run, fast):** loads the subset from Drive and trains a ResNet-18 with a multi-label head.

**Split design:** official `train` → `train`, official `validation` → `val`, official `test` → split 50/50 (seeded) into `test_public` / `test_hidden`. Reusing the official geographic split assignment means our hidden set inherits reBEN's low spatial correlation with training data — better than any random re-split we could make.

**Runtime → T4 GPU** before starting. For the PREP run, also make sure you have a stable connection.

## 1. Configuration

In [ ]:
import os, sys, io, json, csv, random, shutil, zipfile, hashlib, time
from pathlib import Path
from collections import defaultdict

CFG = {
    "seed": 42,
    "split_version": "v1",
    "subset_fraction": 0.10,
    "benchmark": "bigearthnet",
    "model_name": "resnet18-baseline",

    # RGB rendering: reflectance DNs clipped to [0, RGB_CLIP] then scaled to [0, 255].
    # FIXED for the benchmark — changing it changes the data, so it would need a new split_version.
    "rgb_clip": 3000,

    # training
    "img_size": 120,
    "batch_size": 128,
    "epochs": 12,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "num_workers": 2,
}

# Canonical 19-class nomenclature, ALPHABETICAL — prob_00..prob_18 map to this order.
BENCHMARK_CLASSES = [
    "Agro-forestry areas",
    "Arable land",
    "Beaches, dunes, sands",
    "Broad-leaved forest",
    "Coastal wetlands",
    "Complex cultivation patterns",
    "Coniferous forest",
    "Industrial or commercial units",
    "Inland waters",
    "Inland wetlands",
    "Land principally occupied by agriculture, with significant areas of natural vegetation",
    "Marine waters",
    "Mixed forest",
    "Moors, heathland and sclerophyllous vegetation",
    "Natural grassland and sparsely vegetated areas",
    "Pastures",
    "Permanent crops",
    "Transitional woodland, shrub",
    "Urban fabric",
]
N_CLASSES = len(BENCHMARK_CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(BENCHMARK_CLASSES)}

META_URL = "https://zenodo.org/records/10891137/files/metadata.parquet?download=1"
S2_URL   = "https://zenodo.org/records/10891137/files/BigEarthNet-S2.tar.zst?download=1"

random.seed(CFG["seed"])
print(f"{N_CLASSES} classes, subset fraction {CFG['subset_fraction']}, seed {CFG['seed']}")

## 2. Environment

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR  = Path("/content/drive/MyDrive/eobench/bigearthnet")  # persistent
    LOCAL_DIR = Path("/content/ben")                                  # fast, ephemeral
else:
    WORK_DIR, LOCAL_DIR = Path("./eobench_work/bigearthnet"), Path("./ben_local")

CKPT_DIR, OUT_DIR = WORK_DIR / "checkpoints", WORK_DIR / "outputs"
for d in (CKPT_DIR, OUT_DIR, LOCAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

SUBSET_ZIP    = WORK_DIR / f"ben_s2_rgb_subset_{CFG['split_version']}.zip"
MANIFEST_PATH = WORK_DIR / f"split_manifest_{CFG['split_version']}.csv"

!pip -q install zstandard tifffile torchmetrics

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import torchmetrics

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| prep already done:" , SUBSET_ZIP.exists() and MANIFEST_PATH.exists())

## 3. PREP — build the subset (one-time; auto-skipped afterwards)

Downloads the official archive to Colab's **local disk** (never Drive — it wouldn't fit), selects a seeded 10% per official split from `metadata.parquet`, and stream-extracts just those patches' B04/B03/B02 bands into PNGs. Nothing but the small subset zip + manifest is kept.

Needs ~65 GB free local disk and a stable session for ~1–2 hours. If the session dies mid-download, just rerun — the download resumes from where it stopped.

In [ ]:
def http_download_resumable(url, dest: Path, chunk=1 << 22):
    import urllib.request
    done = dest.stat().st_size if dest.exists() else 0
    req = urllib.request.Request(url, headers={"User-Agent": "eobench/0.1",
                                               **({"Range": f"bytes={done}-"} if done else {})})
    try:
        r = urllib.request.urlopen(req)
    except urllib.error.HTTPError as e:
        if e.code == 416:   # already complete
            return
        raise
    total = done + int(r.headers.get("Content-Length", 0))
    mode = "ab" if done else "wb"
    with open(dest, mode) as f:
        t0 = time.time()
        while True:
            buf = r.read(chunk)
            if not buf:
                break
            f.write(buf); done += len(buf)
            mb, tot = done / 1e9, total / 1e9
            rate = done / max(time.time() - t0, 1) / 1e6
            print(f"\r  {mb:6.2f} / {tot:.2f} GB  ({rate:5.1f} MB/s)", end="")
    print()


def detect_columns(df):
    """Find patch-id / labels / split columns without hardcoding names."""
    id_col = split_col = labels_col = None
    for c in df.columns:
        sample = df[c].iloc[0]
        vals = set(df[c].unique()[:10].tolist()) if df[c].dtype == object else set()
        if split_col is None and vals and vals <= {"train", "validation", "test"}:
            split_col = c
        elif labels_col is None and isinstance(sample, (list, np.ndarray)):
            labels_col = c
        elif id_col is None and isinstance(sample, str) and sample.startswith("S2"):
            id_col = c
    assert all([id_col, labels_col, split_col]), \
        f"Could not detect columns in {list(df.columns)} — inspect the parquet and adjust."
    return id_col, labels_col, split_col


def run_prep():
    import tifffile, zstandard, tarfile
    from PIL import Image

    free_gb = shutil.disk_usage(LOCAL_DIR).free / 1e9
    assert free_gb > 65, f"Only {free_gb:.0f} GB free local disk — need ~65 GB for the archive."

    # --- 1. metadata + deterministic selection ---
    meta_path = LOCAL_DIR / "metadata.parquet"
    if not meta_path.exists():
        print("Downloading metadata.parquet ...")
        http_download_resumable(META_URL, meta_path)
    df = pd.read_parquet(meta_path)
    id_col, labels_col, split_col = detect_columns(df)
    print(f"metadata: {len(df)} patches | columns: id={id_col}, labels={labels_col}, split={split_col}")

    selected = {}   # patch_id -> (labels, our_split)
    rng_pub = random.Random(f"{CFG['seed']}-test-halving")
    for official in ("train", "validation", "test"):
        ids = sorted(df.loc[df[split_col] == official, id_col].tolist())
        rng = random.Random(f"{CFG['seed']}-{official}")
        keep = rng.sample(ids, round(len(ids) * CFG["subset_fraction"]))
        lab = df.set_index(id_col).loc[keep, labels_col]
        for pid in keep:
            labels = [str(x) for x in lab.loc[pid]]
            assert all(l in CLASS_TO_IDX for l in labels), f"Unknown label in {pid}: {labels}"
            if official == "train":
                ours = "train"
            elif official == "validation":
                ours = "val"
            else:
                ours = "test_public" if rng_pub.random() < 0.5 else "test_hidden"
            selected[pid] = (labels, ours)
    from collections import Counter
    print("subset:", dict(Counter(v[1] for v in selected.values())), f"total {len(selected)}")

    # --- 2. archive download (resumable) ---
    tar_path = LOCAL_DIR / "BigEarthNet-S2.tar.zst"
    print("Downloading BigEarthNet-S2.tar.zst (~59 GiB — grab a coffee) ...")
    http_download_resumable(S2_URL, tar_path)

    # --- 3. one streaming pass: extract B02/B03/B04 for selected patches ---
    png_dir = LOCAL_DIR / "subset_png"
    png_dir.mkdir(exist_ok=True)
    wanted_bands = ("B02", "B03", "B04")
    bands_buf: dict[str, dict] = defaultdict(dict)
    saved, t0 = 0, time.time()

    def flush(pid):
        nonlocal saved
        b = bands_buf.pop(pid)
        rgb = np.stack([b["B04"], b["B03"], b["B02"]], axis=-1).astype(np.float32)
        rgb = np.clip(rgb, 0, CFG["rgb_clip"]) / CFG["rgb_clip"] * 255.0
        Image.fromarray(rgb.astype(np.uint8)).save(png_dir / f"{pid}.png")
        saved += 1
        if saved % 2000 == 0:
            print(f"\r  saved {saved}/{len(selected)} patches ({time.time()-t0:.0f}s)", end="")

    with open(tar_path, "rb") as fh:
        stream = zstandard.ZstdDecompressor().stream_reader(fh)
        with tarfile.open(fileobj=stream, mode="r|") as tar:
            for member in tar:
                if not member.isfile() or not member.name.endswith(".tif"):
                    continue
                fname = Path(member.name).name          # <patch_id>_<band>.tif
                stem = fname[:-4]
                pid, _, band = stem.rpartition("_")
                if band not in wanted_bands or pid not in selected:
                    continue
                data = tar.extractfile(member).read()
                bands_buf[pid][band] = tifffile.imread(io.BytesIO(data))
                if len(bands_buf[pid]) == 3:
                    flush(pid)
    print(f"\nExtracted {saved} complete patches.")
    incomplete = [pid for pid in selected if not (png_dir / f"{pid}.png").exists()]
    if incomplete:
        print(f"WARNING: {len(incomplete)} selected patches had missing bands — dropped from the benchmark.")
        for pid in incomplete:
            selected.pop(pid)

    # --- 4. manifest + subset zip -> Drive ---
    rows = [{"sample_id": pid, "relpath": f"{pid}.png",
             "labels": "|".join(sorted(labels)), "split": ours}
            for pid, (labels, ours) in sorted(selected.items())]
    with open(MANIFEST_PATH, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["sample_id", "relpath", "labels", "split"])
        w.writeheader(); w.writerows(rows)
    fp = hashlib.sha256("\n".join(f"{r['sample_id']},{r['split']}" for r in rows).encode()).hexdigest()[:16]
    print("manifest ->", MANIFEST_PATH, "| split fingerprint:", fp)

    print("Zipping subset to Drive ...")
    with zipfile.ZipFile(SUBSET_ZIP, "w", zipfile.ZIP_STORED) as z:  # PNGs are already compressed
        for png in sorted(png_dir.glob("*.png")):
            z.write(png, png.name)
    print(f"done -> {SUBSET_ZIP} ({SUBSET_ZIP.stat().st_size/1e9:.2f} GB)")
    tar_path.unlink(missing_ok=True)   # reclaim the 59 GB


if SUBSET_ZIP.exists() and MANIFEST_PATH.exists():
    print("Subset already prepared — skipping PREP.")
else:
    run_prep()

## 4. Load the subset

Copies the zip from Drive to local disk (Drive I/O is slow for many small reads) and loads the manifest.

In [ ]:
IMG_DIR = LOCAL_DIR / "images"
if not IMG_DIR.exists() or not any(IMG_DIR.iterdir()):
    IMG_DIR.mkdir(exist_ok=True)
    print("Unzipping subset to local disk ...")
    with zipfile.ZipFile(SUBSET_ZIP) as z:
        z.extractall(IMG_DIR)

manifest = list(csv.DictReader(open(MANIFEST_PATH, newline="")))
from collections import Counter
print(dict(Counter(r["split"] for r in manifest)), "| total:", len(manifest))

# class prevalence in train (for pos_weight — BigEarthNet is heavily imbalanced)
train_rows = [r for r in manifest if r["split"] == "train"]
counts = np.zeros(N_CLASSES)
for r in train_rows:
    for l in r["labels"].split("|"):
        counts[CLASS_TO_IDX[l]] += 1
pos_weight = torch.tensor(np.clip((len(train_rows) - counts) / np.maximum(counts, 1), 1, 20),
                          dtype=torch.float32)
for c, n in sorted(zip(BENCHMARK_CLASSES, counts.astype(int)), key=lambda t: t[1]):
    print(f"  {n:>6}  {c}")

## 5. Dataset, model, training (checkpoint + auto-resume)

In [ ]:
from PIL import Image

NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ToTensor(), NORM,
])
eval_tf = transforms.Compose([transforms.ToTensor(), NORM])

class BenSubset(Dataset):
    def __init__(self, rows, split, tf, with_labels=True):
        self.rows = [r for r in rows if r["split"] == split]
        self.tf, self.with_labels = tf, with_labels
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = self.tf(Image.open(IMG_DIR / r["relpath"]).convert("RGB"))
        if not self.with_labels:
            return img, r["sample_id"]
        y = torch.zeros(N_CLASSES)
        for l in r["labels"].split("|"):
            y[CLASS_TO_IDX[l]] = 1.0
        return img, y

def loader(split, tf, shuffle, with_labels=True):
    return DataLoader(BenSubset(manifest, split, tf, with_labels),
                      batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=True)

train_loader, val_loader = loader("train", train_tf, True), loader("val", eval_tf, False)
pub_loader = loader("test_public", eval_tf, False)

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
scaler    = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
map_metric = torchmetrics.classification.MultilabelAveragePrecision(
    num_labels=N_CLASSES, average="macro").to(DEVICE)

LAST_CKPT, BEST_CKPT = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt"
start_epoch, best_val_map = 0, 0.0
if LAST_CKPT.exists():
    ck = torch.load(LAST_CKPT, map_location=DEVICE)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optim"])
    scheduler.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, best_val_map = ck["epoch"] + 1, ck["best_val_map"]
    print(f"Resumed from epoch {ck['epoch']} (best val mAP {best_val_map:.4f})")

@torch.no_grad()
def eval_map(loader):
    model.eval(); map_metric.reset()
    for x, y in loader:
        logits = model(x.to(DEVICE, non_blocking=True))
        map_metric.update(logits, y.to(DEVICE, non_blocking=True).long())
    return map_metric.compute().item()

for epoch in range(start_epoch, CFG["epochs"]):
    model.train(); t0 = time.time(); running = 0.0
    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item()
        if step % 50 == 0:
            print(f"\r  epoch {epoch+1}/{CFG['epochs']}  step {step}/{len(train_loader)}"
                  f"  loss {running/(step+1):.4f}", end="")
    scheduler.step()
    val_map = eval_map(val_loader)
    print(f"\repoch {epoch+1}/{CFG['epochs']}  loss {running/len(train_loader):.4f}"
          f"  val mAP(macro) {val_map:.4f}  ({time.time()-t0:.0f}s)      ")
    state = {"epoch": epoch, "model": model.state_dict(), "optim": optimizer.state_dict(),
             "sched": scheduler.state_dict(), "scaler": scaler.state_dict(),
             "best_val_map": max(best_val_map, val_map), "cfg": CFG}
    torch.save(state, LAST_CKPT)
    if val_map > best_val_map:
        best_val_map = val_map
        torch.save(state, BEST_CKPT)
        print(f"  ↑ new best ({best_val_map:.4f}) — saved best.pt")
print(f"\nBest val mAP(macro): {best_val_map:.4f}")

## 6. Sanity check on `test_public`

In [ ]:
ck = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ck["model"])
print(f"test_public mAP(macro): {eval_map(pub_loader):.4f}")
print("Expect test_hidden within ~±1 point of this (both halves of the official test split).")

## 7. Inference on `test_hidden` → probabilities CSV

Required schema: header `sample_id,prob_00,...,prob_18` where `prob_XX` is the **sigmoid probability** for `BENCHMARK_CLASSES[XX]` (alphabetical order). Values in [0, 1].

In [ ]:
hidden_loader = loader("test_hidden", eval_tf, shuffle=False, with_labels=False)

rows = []
model.eval()
with torch.no_grad():
    for x, sample_ids in hidden_loader:
        probs = torch.sigmoid(model(x.to(DEVICE, non_blocking=True))).clamp(0, 1).cpu().numpy()
        rows += [(sid, p) for sid, p in zip(sample_ids, probs)]
rows.sort(key=lambda t: t[0])

PRED_PATH = OUT_DIR / f"predictions_{CFG['model_name']}_{CFG['split_version']}.csv"
with open(PRED_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["sample_id"] + [f"prob_{i:02d}" for i in range(N_CLASSES)])
    for sid, p in rows:
        w.writerow([sid] + [f"{v:.6f}" for v in p])
print(f"Wrote {len(rows)} rows -> {PRED_PATH}")

## 8. Local validation (mirror of the platform validator)

In [ ]:
def validate_predictions(pred_path, manifest_rows, n_classes):
    expected_ids = {r["sample_id"] for r in manifest_rows if r["split"] == "test_hidden"}
    errors = []
    with open(pred_path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        if header != ["sample_id"] + [f"prob_{i:02d}" for i in range(n_classes)]:
            return ["Bad header"]
        seen = set()
        for ln, row in enumerate(reader, start=2):
            if len(row) != n_classes + 1:
                errors.append(f"Line {ln}: wrong column count"); continue
            sid = row[0]
            if sid in seen:
                errors.append(f"Duplicate sample_id {sid}")
            seen.add(sid)
            try:
                if any(not (0.0 <= float(v) <= 1.0) for v in row[1:]):
                    errors.append(f"Line {ln}: probability outside [0,1]")
            except ValueError:
                errors.append(f"Line {ln}: non-numeric value")
    if expected_ids - seen: errors.append(f"{len(expected_ids - seen)} missing hidden id(s)")
    if seen - expected_ids: errors.append(f"{len(seen - expected_ids)} unknown id(s)")
    return errors

errs = validate_predictions(PRED_PATH, manifest, N_CLASSES)
print("✔ valid — ready to submit" if not errs else "VALIDATION FAILED:")
for e in errs: print("  ✗", e)

## 9. Submit

**Platform setup first (one-time):** copy `split_manifest_v1.csv` from `Drive/eobench/bigearthnet/` into the platform repo at `data/ground_truth/bigearthnet/split_manifest_v1.csv`, then run `make seed`.

**Option A:** download the CSV below and upload on the **BigEarthNet** page of your instance.
**Option B:** direct API submission over an ngrok tunnel.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(PRED_PATH))
else:
    print(PRED_PATH.resolve())

In [ ]:
API_BASE, API_TOKEN = "", ""   # e.g. "https://xxx.ngrok-free.app", "eob_..."
if API_BASE and API_TOKEN:
    import requests
    with open(PRED_PATH, "rb") as f:
        r = requests.post(f"{API_BASE}/benchmarks/bigearthnet/submissions",
                          headers={"Authorization": f"Bearer {API_TOKEN}"},
                          files={"file": (PRED_PATH.name, f, "text/csv")})
    r.raise_for_status()
    print("Submission:", r.json()["id"], r.json()["status"])
else:
    print("Use Option A (web upload).")

## 10. Model card starter

- **Architecture:** ResNet-18, multi-label head (19 sigmoid outputs), BCEWithLogits + pos_weight
- **Backbone:** ResNet-18 · **Params:** ~11.2 M · **Input:** 120×120 RGB (B04/B03/B02, clip-3000 rendering)
- **Pretraining:** ImageNet-1k · **External data:** no
- **Training:** AdamW 3e-4, cosine, 12 epochs, batch 128, AMP, H/V flips · **Hardware:** Colab T4
- **Data:** BigEarthNet v2.0 (reBEN) 10% RGB subset, official geographic splits, `split_v1`, seed 42

**Honest caveat for the benchmark description:** RGB-only at 10% is *not* comparable to published full-dataset, all-bands reBEN numbers — expect a macro mAP well below the ~0.7+ that big multispectral models reach. That's fine: entries on *this* leaderboard compare against each other under identical conditions.

**Beating the baseline:** all 10+ bands (change PREP to keep more bands, new split_version), ResNet-50, threshold tuning per class, asymmetric loss, SSL4EO-pretrained backbones.